# Lab 4 — Particionamento por ano e mês

## Objetivo

Este laboratório demonstra como organizar fisicamente uma base de transações em partições.

A base será convertida de CSV para Parquet e separada por ano e mês. Essa estratégia permite que consultas temporais leiam somente as pastas necessárias, reduzindo o volume de dados processado.

O particionamento físico será realizado pelo DuckDB, seguindo o padrão Hive de diretórios, como `year=2023/month=1`.

In [1]:
from pathlib import Path
import shutil
import time
import statistics
import duckdb
import pandas as pd

pasta_projeto = Path.cwd().resolve()

if not (pasta_projeto / "dados" / "raw").exists():
    for pasta_pai in pasta_projeto.parents:
        if (pasta_pai / "dados" / "raw").exists():
            pasta_projeto = pasta_pai
            break

arquivo_transacoes = (
    pasta_projeto
    / "dados"
    / "raw"
    / "sqoop_import"
    / "transactions"
    / "transactions_from_db.csv"
)

pasta_particionada = (
    pasta_projeto
    / "dados"
    / "silver"
    / "transactions_particionado"
)

pasta_lab04 = (
    pasta_projeto
    / "dia2_transformacao"
    / "lab04_particoes"
)

arquivo_banco = pasta_lab04 / "particionamento.duckdb"

assert arquivo_transacoes.exists(), (
    f"Arquivo não encontrado: {arquivo_transacoes}"
)

print("Transações:", arquivo_transacoes)
print("Destino particionado:", pasta_particionada)

Transações: C:\BigData\bigdata-curso-gabriel\dados\raw\sqoop_import\transactions\transactions_from_db.csv
Destino particionado: C:\BigData\bigdata-curso-gabriel\dados\silver\transactions_particionado


In [2]:
conexao = duckdb.connect(str(arquivo_banco))

caminho_transacoes_sql = (
    arquivo_transacoes.as_posix().replace("'", "''")
)

conexao.execute(f"""
    CREATE OR REPLACE VIEW raw_transactions AS
    SELECT
        transaction_id,
        customer_id,
        amount,
        transaction_type,
        TRY_CAST(timestamp AS TIMESTAMP) AS transaction_timestamp,
        status,
        risk_score,
        TRY_CAST(is_fraud AS BOOLEAN) AS is_fraud,
        YEAR(TRY_CAST(timestamp AS TIMESTAMP)) AS year,
        MONTH(TRY_CAST(timestamp AS TIMESTAMP)) AS month
    FROM read_csv_auto(
        '{caminho_transacoes_sql}',
        header = true,
        all_varchar = true
    )
""")

quantidade_raw = conexao.execute("""
    SELECT COUNT(*)
    FROM raw_transactions
""").fetchone()[0]

print("Total de transações:", quantidade_raw)

Total de transações: 100000


In [3]:
validacao_datas = conexao.execute("""
    SELECT
        COUNT(*) AS total_linhas,
        COUNT(transaction_timestamp) AS datas_validas,
        COUNT(*) - COUNT(transaction_timestamp) AS datas_invalidas,
        MIN(transaction_timestamp) AS primeira_data,
        MAX(transaction_timestamp) AS ultima_data
    FROM raw_transactions
""").df()

validacao_datas

,total_linhas,datas_validas,datas_invalidas,primeira_data,ultima_data
0,100000,100000,0,2023-01-01,2024-12-28


In [4]:
particoes_planejadas = conexao.execute("""
    SELECT
        year,
        month,
        COUNT(*) AS quantidade_transacoes
    FROM raw_transactions
    GROUP BY year, month
    ORDER BY year, month
""").df()

print("Quantidade de partições:", len(particoes_planejadas))
particoes_planejadas

Quantidade de partições: 24


,year,month,quantidade_transacoes
0,2023,1,4276
1,2023,2,3787
2,2023,3,4266
3,2023,4,4231
4,2023,5,4353
5,2023,6,4130
6,2023,7,4177
7,2023,8,4157
8,2023,9,4061
9,2023,10,4351


In [5]:
if pasta_particionada.exists():
    shutil.rmtree(pasta_particionada)

pasta_particionada.parent.mkdir(
    parents=True,
    exist_ok=True
)

caminho_particionado_sql = (
    pasta_particionada.as_posix().replace("'", "''")
)

conexao.execute(f"""
    COPY (
        SELECT *
        FROM raw_transactions
    )
    TO '{caminho_particionado_sql}'
    (
        FORMAT PARQUET,
        PARTITION_BY (year, month),
        COMPRESSION ZSTD
    )
""")

print("Dados gravados em Parquet particionado.")

Dados gravados em Parquet particionado.


In [6]:
pastas_particoes = sorted(
    pasta
    for pasta in pasta_particionada.glob("year=*/month=*")
    if pasta.is_dir()
)

estrutura_particoes = []

for pasta in pastas_particoes:
    arquivos_parquet = list(pasta.glob("*.parquet"))

    estrutura_particoes.append({
        "Partição": str(
            pasta.relative_to(pasta_particionada)
        ),
        "Arquivos Parquet": len(arquivos_parquet),
        "Tamanho em KB": round(
            sum(
                arquivo.stat().st_size
                for arquivo in arquivos_parquet
            ) / 1024,
            2
        )
    })

estrutura_particoes_df = pd.DataFrame(estrutura_particoes)

print("Pastas de partição encontradas:", len(estrutura_particoes_df))
estrutura_particoes_df

Pastas de partição encontradas: 24


,Partição,Arquivos Parquet,Tamanho em KB
0,year=2023\month=1,1,105.14
1,year=2023\month=10,1,106.81
2,year=2023\month=11,1,102.74
3,year=2023\month=12,1,103.30
4,year=2023\month=2,1,93.07
5,year=2023\month=3,1,104.62
6,year=2023\month=4,1,103.79
7,year=2023\month=5,1,106.59
8,year=2023\month=6,1,101.36
9,year=2023\month=7,1,102.63


In [7]:
padrao_parquet_sql = (
    (pasta_particionada / "**" / "*.parquet")
    .as_posix()
    .replace("'", "''")
)

quantidade_parquet = conexao.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{padrao_parquet_sql}',
        hive_partitioning = true
    )
""").fetchone()[0]

validacao_particionamento = pd.DataFrame({
    "Indicador": [
        "Linhas na Raw",
        "Linhas no Parquet",
        "Diferença",
        "Quantidade de partições"
    ],
    "Resultado": [
        quantidade_raw,
        quantidade_parquet,
        quantidade_raw - quantidade_parquet,
        len(pastas_particoes)
    ]
})

validacao_particionamento

,Indicador,Resultado
0,Linhas na Raw,100000
1,Linhas no Parquet,100000
2,Diferença,0
3,Quantidade de partições,24


In [8]:
consulta_csv = """
    SELECT COUNT(*)
    FROM raw_transactions
    WHERE month = 1
"""

consulta_parquet = f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{padrao_parquet_sql}',
        hive_partitioning = true
    )
    WHERE month = 1
"""

janeiro_csv = conexao.execute(
    consulta_csv
).fetchone()[0]

janeiro_parquet = conexao.execute(
    consulta_parquet
).fetchone()[0]

comparacao_resultado = pd.DataFrame({
    "Origem": [
        "CSV sem particionamento",
        "Parquet particionado"
    ],
    "Transações de janeiro": [
        janeiro_csv,
        janeiro_parquet
    ]
})

comparacao_resultado

,Origem,Transações de janeiro
0,CSV sem particionamento,8461
1,Parquet particionado,8461


In [9]:
def medir_tempo(consulta, repeticoes=5):
    tempos = []

    for _ in range(repeticoes):
        inicio = time.perf_counter()
        conexao.execute(consulta).fetchall()
        fim = time.perf_counter()
        tempos.append(fim - inicio)

    return tempos


tempos_csv = medir_tempo(consulta_csv)
tempos_parquet = medir_tempo(consulta_parquet)

comparacao_tempo = pd.DataFrame({
    "Consulta": [
        "CSV sem particionamento",
        "Parquet particionado"
    ],
    "Mediana em segundos": [
        statistics.median(tempos_csv),
        statistics.median(tempos_parquet)
    ],
    "Menor tempo em segundos": [
        min(tempos_csv),
        min(tempos_parquet)
    ],
    "Repetições": [len(tempos_csv), len(tempos_parquet)]
})

comparacao_tempo

,Consulta,Mediana em segundos,Menor tempo em segundos,Repetições
0,CSV sem particionamento,0.240180,0.236241,5
1,Parquet particionado,0.014753,0.014632,5


In [10]:
particoes_de_janeiro = estrutura_particoes_df[
    estrutura_particoes_df["Partição"].str.endswith("month=1")
]

print(
    "Para consultar janeiro, são necessárias",
    len(particoes_de_janeiro),
    "partições:"
)

particoes_de_janeiro

Para consultar janeiro, são necessárias 2 partições:


,Partição,Arquivos Parquet,Tamanho em KB
0,year=2023\month=1,1,105.14
12,year=2024\month=1,1,102.72


In [11]:
conexao.close()

print("Conexão encerrada.")
print("Lab 4 executado com sucesso.")

Conexão encerrada.
Lab 4 executado com sucesso.


## Conclusão

As 100.000 transações foram lidas da camada Raw, e as colunas de ano e mês foram derivadas a partir da data de cada transação. Em seguida, os dados foram convertidos do formato CSV para Parquet e organizados fisicamente em partições anuais e mensais.

A validação demonstrou que a quantidade total de registros permaneceu inalterada após o particionamento, sem perda de transações. As pastas criadas seguem o padrão Hive, como `year=2023/month=1`.

A estratégia por ano e mês é adequada para consultas como acompanhamento mensal do volume financeiro, análise da evolução das fraudes e comparação entre períodos. Nesse cenário, o mecanismo pode ignorar partições que não atendem ao filtro, reduzindo a leitura desnecessária.

Em uma base pequena, a diferença de tempo pode ser pouco perceptível ou variar entre execuções. Em grandes volumes, a redução da quantidade de arquivos e registros examinados tende a produzir benefícios mais relevantes.

No Hive, quando partições já existem fisicamente, mas ainda não estão registradas no catálogo, o comando `MSCK REPAIR TABLE` pode ser utilizado para descobri-las e atualizar os metadados. No DuckDB, as partições no padrão Hive são reconhecidas diretamente durante a leitura dos arquivos Parquet.